# Zaskaleta AI Twin — Free Colab GPU
Український текст → MMS Ukrainian TTS → OpenVoice V2 → MuseTalk 1.5 → MP4 9:16.

Master-фото (5–6 шт.) і Master-голос беруться з Google Drive і не завантажуються в GitHub.

In [ ]:
import torch, subprocess, os
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU не активний. Runtime → Change runtime type → T4 GPU')
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
!rm -rf /content/zaskaleta-ai-twin-colab
!git clone --depth 1 https://github.com/sergokharkov/zaskaleta-ai-twin-colab.git /content/zaskaleta-ai-twin-colab
WORKER='/content/zaskaleta-ai-twin-colab/worker'
MUSETALK='/content/MuseTalk'
VENV_DIR='/content/ai-twin-py311'
PYTHON_BIN=f'{VENV_DIR}/bin/python'
print('✅ Public AI Twin code loaded')

In [ ]:
env=os.environ.copy()
env['APP_DIR']=WORKER
env['MUSETALK_ROOT']=MUSETALK
env['VENV_DIR']=VENV_DIR
result=subprocess.run(['bash', '-x', f'{WORKER}/install_gpu_engines.sh'], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
log=result.stdout or ''
print(log)
if result.returncode != 0:
    print('\n===== INSTALLER FAILED =====')
    print('exit code:', result.returncode)
    print('\n===== LAST 120 LINES =====')
    print('\n'.join(log.splitlines()[-120:]))
    raise RuntimeError(f'GPU installer failed with code {result.returncode}')
print('✅ OpenVoice + MuseTalk installed in Python 3.11')
subprocess.run([PYTHON_BIN, '-c', 'import sys, torch; print(sys.version); print("CUDA in venv:", torch.cuda.is_available())'], check=True)

## 1. Підключіть Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_FOLDER = Path('/content/drive/MyDrive/Zaskaleta_AI_Twin')
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)
print('✅ Google Drive mounted')
print('📁 Робоча папка:', DRIVE_FOLDER)

## 2. Виберіть 5–6 MASTER PHOTO і MASTER VOICE
Покладіть 5–6 фото та голосовий WAV/MP3 у **MyDrive/Zaskaleta_AI_Twin**. Потім запустіть клітинку нижче.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
image_ext={'.jpg','.jpeg','.png','.webp'}
audio_ext={'.wav','.mp3','.m4a','.flac'}
photos=sorted([p for p in DRIVE_FOLDER.iterdir() if p.is_file() and p.suffix.lower() in image_ext])
voices=sorted([p for p in DRIVE_FOLDER.iterdir() if p.is_file() and p.suffix.lower() in audio_ext])
if len(photos) < 5:
    raise FileNotFoundError(f'Знайдено лише {len(photos)} фото. Додайте щонайменше 5 фото у {DRIVE_FOLDER}.')
if not voices:
    raise FileNotFoundError(f'У {DRIVE_FOLDER} немає голосу. Додайте WAV/MP3/M4A/FLAC.')
photo_multi=widgets.SelectMultiple(options=[(p.name,str(p)) for p in photos], description='PHOTOS:', rows=min(10,len(photos)), layout=widgets.Layout(width='95%'))
voice_pick=widgets.Dropdown(options=[(p.name,str(p)) for p in voices], description='VOICE:', layout=widgets.Layout(width='95%'))
display(photo_multi, voice_pick)
print('Оберіть рівно 5 або 6 фото (довге натискання/CTRL залежно від пристрою), потім запускайте наступну клітинку.')

In [ ]:
MASTER_PHOTOS=list(photo_multi.value)
VOICE=voice_pick.value
if len(MASTER_PHOTOS) not in (5,6):
    raise ValueError(f'Потрібно вибрати 5 або 6 фото. Зараз вибрано: {len(MASTER_PHOTOS)}')
for p in MASTER_PHOTOS:
    if not Path(p).is_file(): raise FileNotFoundError(p)
if not Path(VOICE).is_file(): raise FileNotFoundError(VOICE)
print('✅ MASTER PHOTOS:')
for i,p in enumerate(MASTER_PHOTOS,1): print(f'{i}. {p}')
print('✅ VOICE =', VOICE)

## 3. Виберіть основне фото для цього ролика
MuseTalk генерує один ролик з одного вихідного фото. Інші 4–5 фото залишаються в наборі Master для наступних роликів і різних ракурсів.

In [ ]:
primary_pick=widgets.Dropdown(options=[(Path(p).name,p) for p in MASTER_PHOTOS], description='PRIMARY:', layout=widgets.Layout(width='95%'))
display(primary_pick)
print('Виберіть основне фото для поточного відео.')

In [ ]:
PHOTO=primary_pick.value
print('✅ PRIMARY PHOTO =', PHOTO)

## 4. Введіть український текст

In [ ]:
TEXT = 'Привіт. Це тест мого AI-двійника. Я говорю українською своїм голосом.' #@param {type:"string"}
SCRIPT='/content/zaskaleta_script.txt'
Path(SCRIPT).write_text(TEXT, encoding='utf-8')
print(TEXT)

In [ ]:
VOICE_OUT='/content/zaskaleta_voice.wav'
subprocess.run([PYTHON_BIN, f'{WORKER}/voice_mms_openvoice.py', '--script', SCRIPT, '--voice', VOICE, '--output', VOICE_OUT, '--language', 'uk'], check=True)
print('✅ Voice ready:', VOICE_OUT)

In [ ]:
RAW_VIDEO='/content/zaskaleta_raw.mp4'
env=os.environ.copy(); env['MUSETALK_ROOT']=MUSETALK
subprocess.run([PYTHON_BIN, f'{WORKER}/lipsync_musetalk.py', '--photo', PHOTO, '--audio', VOICE_OUT, '--output', RAW_VIDEO], env=env, check=True)
print('✅ Lip-sync ready:', RAW_VIDEO)

In [ ]:
FINAL='/content/Zaskaleta_AI_Twin_9x16.mp4'
vf='scale=1080:1920:force_original_aspect_ratio=decrease,pad=1080:1920:(ow-iw)/2:(oh-ih)/2,setsar=1'
subprocess.run(['ffmpeg','-y','-i',RAW_VIDEO,'-vf',vf,'-c:v','libx264','-preset','medium','-crf','19','-c:a','aac','-b:a','192k','-movflags','+faststart',FINAL], check=True)
print('✅ FINAL:', FINAL)

In [ ]:
from IPython.display import Video, display
display(Video(FINAL, embed=True, width=360))
from google.colab import files
files.download(FINAL)